# Message Trainer

> Fill in a module description here

In [ ]:
#| default_exp trainers.msg_trainer

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import torch
import os
import torch.nn as nn
import torchvision.utils as vutils
import wandb
from c3jepa_wm.utils.checkpointer import RetrospectiveCheckpointer
import hydra
from pathlib import Path

## Pure Torch Trainer

In [ ]:
#| export

class VQVAETrainer:

    def __init__(self, cfg, model, device, slurm_jobid= None):
        self.cfg = cfg
        self.model = model.to(device)
        self.slurm_jobid = slurm_jobid if slurm_jobid else "default_job"
        self.device = device
        self.save_dir = Path(hydra.utils.to_absolute_path(cfg.logging_params.save_dir))
        self.save_dir.mkdir(exist_ok=True, parents=True)

        # Match the optimizer configuration from your original script
        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=3e-4)
        self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer, mode='min', factor=0.5, patience=5, verbose=True
        )

        self.ck_pointer = RetrospectiveCheckpointer(cfg= cfg, slurm_jobid= self.slurm_jobid, agent_id= "vqvae_trainer", rank= 0, n_best=3)

        # Create output directories for visual inspection
        os.makedirs(os.path.join(self.save_dir, "Reconstructions"), exist_ok=True)

    def train_epoch(self, dataloader, epoch):
        self.model.train()
        # self.model.vq_layer.training= True
        total_loss = 0.0

        for batch_idx, batch in enumerate(dataloader):
            # Manually move data to target device (Lightning did this automatically)
            real_img = batch.to(self.device)

            self.optimizer.zero_grad()

            # Forward pass
            recons, input_img, vq_loss, perplexity = self.model(real_img)

            # Call your model's built-in VQ-VAE loss evaluation function
            loss_dict = self.model.loss_function(
                recons,
                input_img,
                vq_loss,
                perplexity,
                M_N=self.cfg.exp_params.get("kld_weight", 1.0),
                batch_idx=batch_idx,
            )

            loss = loss_dict["loss"]
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
            self.optimizer.step()

            total_loss += loss.item()

            # Log step-level metrics directly to WandB
            wandb.log({f"train_{k}": v.item() for k, v in loss_dict.items()})

        avg_loss = total_loss / len(dataloader)
        print(f"Epoch {epoch:02d} | Train Loss: {avg_loss:.4f}")
        return avg_loss

    @torch.no_grad()
    def validate_epoch(self, dataloader, epoch):
        # self.model.vq_layer.training= False
        self.model.eval()
        total_loss = 0.0

        for batch_idx, batch in enumerate(dataloader):
            real_img = batch.to(self.device)

            recons, input_img, vq_loss, perplexity = self.model(real_img)
            loss_dict = self.model.loss_function(
                recons, input_img, vq_loss, perplexity, M_N=1.0, batch_idx=batch_idx
            )

            total_loss += loss_dict["loss"].item()

        avg_loss = total_loss / len(dataloader)
        print(f"Epoch {epoch:02d} | Val Loss:   {avg_loss:.4f}")

        # Log epoch-level validation loss
        wandb.log({"val_loss": avg_loss, "epoch": epoch})
        return avg_loss

    @torch.no_grad()
    def sample_and_save_images(self, test_loader, epoch):
        """Replicates on_validation_end by reconstructing a fixed test batch."""
        self.model.eval()

        # Grab the first batch of images from your test/val loader
        test_input = next(iter(test_loader)).to(self.device)

        # Reconstruct images using VQ-VAE decoder
        recons = self.model.generate(test_input)

        # Save grid image to disk
        recons_path = os.path.join(
            self.save_dir, "Reconstructions", f"recons_Epoch_{epoch}.png"
        )
        vutils.save_image(recons.data, recons_path, normalize=True, nrow=8)

        # Upload the reconstructed image grid directly into WandB panel
        wandb.log(
            {"Visual Reconstructions": wandb.Image(recons_path, caption=f"Epoch {epoch}")}
        )


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()